# 1. Prepare Data

In [9]:
import json
import pandas as pd

# 假設你的資料存為字典 root
with open("english_antonyms/syn_ant_sent_sim.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []
for entry in data:
    word = entry["word"]
    for ant in entry.get("ant_list", []):
        rows.append({
            "word": word,
            "antonym": ant["antonym"],
        })

# 建立 DataFrame 並顯示
syn_ant_sent_sim_df = pd.DataFrame(rows)


antonyms_df = pd.read_csv("english_antonyms/antonyms.csv")
antonyms_df = antonyms_df.drop(columns=["part_of_speech"])


syn_ant_link = "hf://datasets/kh4dien/synonym-antonym/data/train-00000-of-00001.parquet"
syn_ant_df = pd.read_parquet(syn_ant_link)
syn_ant_df = syn_ant_df[syn_ant_df["type"] == "antonym"]
syn_ant_df = syn_ant_df.drop(columns=["type"])

syn_ant_df.columns = ["word", "antonym"]
syn_ant_sent_sim_df.columns = ["word", "antonym"]
antonyms_df.columns = ["word", "antonym"]
antonyms_df.to_csv("english_antonyms/antonyms.csv", index=False, encoding="utf-8")
syn_ant_sent_sim_df.to_csv("english_antonyms/syn_ant_sent_sim.csv", index=False, encoding="utf-8")
syn_ant_df.to_csv("english_antonyms/syn_ant_df.csv", index=False, encoding="utf-8")

# 2. Prepare Data

In [20]:
import pandas as pd
import glob

# 假設所有 CSV 都在某資料夾，例如 ./data/
file_paths = glob.glob("./english_antonyms/*.csv")  # 或具體列出檔名

pairs = []

for path in file_paths:
    df = pd.read_csv(path)
    pair_list = list(zip(df.iloc[:, 0], df.iloc[:, 1]))

    # 過濾掉不是字串的項目
    filtered = [(w, a) for w, a in pair_list if isinstance(w, str) and isinstance(a, str)]
    pairs.extend(filtered)

# 現在 pairs 是 List[Tuple[word, antonym]]
print(pairs[:5])  # 顯示前 5 筆

final_df = pd.DataFrame(pairs, columns=["word", "antonym"])
final_df.to_csv("english_antonyms/all_antonyms.csv", index=False)

[('a la carte', "table d'hote"), ('a posteriori', 'a priori'), ('a posteriori', 'a priori'), ('a priori', 'a posteriori'), ('a priori', 'a posteriori')]


# 3. Tokenize Data

In [22]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

for word, antonym in pairs:
    tokens = tokenizer(word, antonym, return_tensors="pt", padding=True, truncation=True)
print("Tokenization and label alignment completed.")

Tokenization and label alignment completed.
